In [2]:
import os
class Announcement:
    def __init__(self, department, priority, duration, title): 
        """
        Initialize an Announcement object with department, priority, duration (in seconds), and title.
        """
        self.department = department
        self.priority = priority
        self.duration = duration
        self.title = title

    def __str__(self): 
        """
        Return a human-readable string representation of the announcement.
        Example: "CCA - Science Fair (90s)"
        """
        return f"{self.department} - {self.title} ({self.duration}s)"

    def csv_str(self): 
        """
        Return a CSV-formatted string for writing to file.
        Example: "CCA,High,90,Science Fair"
        """
        return f"{self.department},{self.priority},{self.duration},{self.title}"

class Queue:
    def __init__(self): 
        """
        Initialize an empty queue using a Python list.
        """
        self.items = []

    def enqueue(self, item): 
        """
        Add an item to the back of the queue.
        """
        self.items.append(item)

    def dequeue(self): 
        """
        Remove and return the item at the front of the queue, if not empty.
        """
        if not self.is_empty():
            return self.items.pop(0)

    def is_empty(self): 
        """
        Return True if the queue is empty.
        """
        return len(self.items) == 0

    def output(self, file): 
        """
        Write each item in the queue to a file, one per line, using its output() method.
        """
        write_data = ""

        for a in self.items:
            write_data += a.csv_str() + '\n'

        file.write(write_data.strip())

class AnnouncementScheduler:
    def __init__(self, time_limit=300):
        """
        Initialize the scheduler with an optional time limit (in seconds).
        Sets up data structures for queues and scheduled announcements.
        """ 
        self.time_limit = time_limit
        self.deferred_filename = "deferred_announcement.txt"
        self.announcement_filename = "announcement.txt"
        self.archive_filename = "read_announcements_archived.txt"
        self.scheduled = []
        self.total_time = 0
        self.queues = {"High": Queue(), "Medium": Queue(), "Low": Queue()} 
        
    def load_file(self, filename): 
        """
        Load Announcement objects from a CSV-formatted file.
        Returns a list of Announcement objects.
        """
        announcements = []
        
        if os.path.exists(filename):
            file = open(filename, 'r')
            
            for line in file:
                department, priority, duration, title = line.split(',')
                announcements.append(Announcement(department, priority, int(duration), title))

        return announcements

    def load_announcements(self):
        """
        Load announcements from both deferred and new announcement files.
        Returns a combined list of Announcement objects.
        """
        deferred_announcements_list = self.load_file(self.deferred_filename)
        new_announcements_list = self.load_file(self.announcement_filename)

        return deferred_announcements_list + new_announcements_list

    def categorize(self, announcements):
        """
        Sorts announcements into the internal priority queues
        based on their 'priority' attribute.
        """
        for a in announcements:
            self.queues[a.priority].enqueue(a)

    def process(self):
        """
        Processes announcements from high to low priority queues.
        Adds them to the schedule until the time limit is reached.
        Returns a queue containing deferred announcements.
        """
        deferred_queue = Queue()

        for level in ["High", "Medium", "Low"]:
            queue = self.queues[level]

            while not queue.is_empty():
                a = queue.dequeue()

                if a.duration + self.total_time <= self.time_limit:
                    self.scheduled.append(a)
                    self.total_time += a.duration
                else:
                    deferred_queue.enqueue(a)

        return deferred_queue

    def display_schedule(self):
        """
        Prints the list of scheduled announcements and total time used.
        """
        print("Today's Announcements:")
        
        for i in range(len(self.scheduled)):
            print(f"{i}. {self.scheduled[i]}")
            
        print(f"Total Time: {self.total_time}s") 

    def archive_scheduled(self):
        """
        Appends the list of scheduled announcements to the archive file.
        Adds a separator line for readability.
        """
        write = ""

        for a in self.scheduled:
            write += str(a) + "\n"

        write += "=" * 80
        
        archive_file = open(self.archive_filename, 'w')
        archive_file.write(write)
        archive_file.close()

    def save_deferred(self, deferred_queue):
        """
        Writes the deferred announcements to the deferred announcement file
        for processing on the next day.
        """
        deferred_file = open(self.deferred_filename, "w")  # overwrites the file
        deferred_queue.output(deferred_file)
        deferred_file.close()

    def clear_today_file(self):
        """
        Clears the contents of the new announcement file in preparation for the next day.
        """
        open(self.announcement_filename, "w").close()

    def run(self):
        """
        Executes the full announcement scheduling routine:
        1. Load deferred and new announcements
        2. Categorize into priority queues
        3. Schedule announcements within time limit
        4. Display and archive the scheduled announcements
        5. Save unscheduled announcements
        6. Clear today's new announcements
        """
        all_ann = self.load_announcements()
        self.categorize(all_ann)
        deferred = self.process()
        self.display_schedule()
        self.archive_scheduled()
        self.save_deferred(deferred)
        self.clear_today_file()

if __name__ == "__main__":
    scheduler = AnnouncementScheduler()
    scheduler.run()

Today's Announcements:
0. Discipline - Uniform Violation Rules
 (90s)
1. Academics - Exam Briefing
 (150s)
2. Leadership - Student Council Promo
 (45s)
Total Time: 285s
